In [1]:
from sedona.spark import SedonaContext
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder().\
    config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1").\
    config("spark.mongodb.read.connection.uri", "mongodb://mongo-sedona:27017")
    

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-59b41b77-1cc4-472b-a1ea-3d3a9b76e442;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.4.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf

# Reading Postgis table

In [3]:
table_name = "points"

postgresql_url = "jdbc:postgresql://postgis:5432/sedona"

df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", "sedona") \
    .option("password", "postgis") \
    .option("dbtable", "points") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [4]:
df.show()

+-------+--------------------+
|   name|            location|
+-------+--------------------+
|Point A|0101000020E610000...|
|Point B|0101000020E610000...|
|Point C|0101000020E610000...|
|Point A|0101000020E610000...|
|Point B|0101000020E610000...|
|Point C|0101000020E610000...|
+-------+--------------------+



In [5]:
df.selectExpr("name", "ST_GeomFromEWKB(location) AS geom").show()

+-------+-------------+
|   name|         geom|
+-------+-------------+
|Point A|POINT (10 20)|
|Point B|POINT (30 40)|
|Point C|POINT (50 60)|
|Point A|POINT (10 20)|
|Point B|POINT (30 40)|
|Point C|POINT (50 60)|
+-------+-------------+



# Reading MySQL table

In [6]:
database_name = "sedona"
user_name = "sedona"
password = "sedona"

mysql_url = f"jdbc:mysql://mysql-sedona:3306/{database_name}"

df = sedona.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("user", user_name) \
    .option("password", password) \
    .option("dbtable", "points") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()
df.show()

+-------+--------------------+
|   name|            location|
+-------+--------------------+
|Point A|[E6 10 00 00 01 0...|
|Point B|[E6 10 00 00 01 0...|
|Point C|[E6 10 00 00 01 0...|
+-------+--------------------+



In [7]:
df.selectExpr(
    "name",
    "ST_GeomFromMySQL(location) AS geom"
).show(3, False)

+-------+-------------+
|name   |geom         |
+-------+-------------+
|Point A|POINT (20 10)|
|Point B|POINT (40 30)|
|Point C|POINT (60 50)|
+-------+-------------+



# Reading From MongoDB

In [8]:
df = sedona.read \
    .option("database", "sedona") \
    .option("collection", "points") \
    .format("mongodb").load()

In [9]:
df.show()

+--------------------+--------------------+-------+
|                 _id|            location|   name|
+--------------------+--------------------+-------+
|6935bff7ba2e14ea3...|{Point, [-74.006,...|Point A|
|6935bff7ba2e14ea3...|{Point, [-118.243...|Point B|
+--------------------+--------------------+-------+



In [10]:
import pyspark.sql.functions as f
 
df.withColumn("location", f.to_json(f.col("location")))\
    .selectExpr("name", "ST_GeomFromGeoJSON(location) AS geom")\
    .show(2, False)
 

+-------+-------------------------+
|name   |geom                     |
+-------+-------------------------+
|Point A|POINT (-74.006 40.7128)  |
|Point B|POINT (-118.2437 34.0522)|
+-------+-------------------------+



# replicate postgis data

In [11]:
import pyspark.sql.functions as f
import pyspark.sql.types as t
 
data = t.StructType([
    t.StructField("name", t.StringType(), True),
    t.StructField("location", t.StructType([
        t.StructField("srid", t.IntegerType()),
        t.StructField("wkb", t.BinaryType())
    ]))
])
 
before = t.StructField("before", data, True)
 
after = t.StructField("after", data, True)
 
payload = t.StructField(
    "payload", t.StructType([before, after])
)
 
schema = t.StructType([
    payload
])

In [12]:
df = sedona \
  .read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka-sedona:29092") \
  .option("subscribe", "sedona-debezium.public.points") \
  .load()

In [13]:
df.show()

+----+--------------------+--------------------+---------+------+--------------------+-------------+
| key|               value|               topic|partition|offset|           timestamp|timestampType|
+----+--------------------+--------------------+---------+------+--------------------+-------------+
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     0|2025-12-07 17:57:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     1|2025-12-07 17:57:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     2|2025-12-07 17:57:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     3|2025-12-07 17:58:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     4|2025-12-07 17:58:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     5|2025-12-07 17:58:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     6|2025-12-07 17:58:...|     

In [14]:
geometry_df = df.select(
    f.from_json(f.expr("CAST(value AS STRING)"), schema).alias("data")
    )\
    .selectExpr(
        "data.payload.after.name as name",
        "data.payload.after.location.wkb as wkb",
        "data.payload.after.location.srid AS srid"
    )\
    .selectExpr("name", "ST_SetSRID(ST_GeomFromWKB(wkb), srid) AS geom")

In [15]:
geometry_df.show()

+-------+-------------+
|   name|         geom|
+-------+-------------+
|Point A|POINT (10 20)|
|Point B|POINT (30 40)|
|Point C|POINT (50 60)|
|Point A|POINT (10 20)|
|Point B|POINT (30 40)|
|Point C|POINT (50 60)|
|Point A|POINT (10 20)|
|Point B|POINT (30 40)|
|Point C|POINT (50 60)|
+-------+-------------+



# insert some rows

In [16]:
from shapely.geometry import Point
from shapely.wkb import dumps

sedona.createDataFrame([
    {"name": "Point D", "location": dumps(Point(40, 80))},
    {"name": "Point E", "location": dumps(Point(100, 30))},
    {"name": "Point F", "location": dumps(Point(25, 10))}
]).write \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("dbtable", "points") \
    .option("user", "sedona") \
    .option("password", "postgis") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [17]:
# more rows should be available

In [18]:
df.show()

+----+--------------------+--------------------+---------+------+--------------------+-------------+
| key|               value|               topic|partition|offset|           timestamp|timestampType|
+----+--------------------+--------------------+---------+------+--------------------+-------------+
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     0|2025-12-07 17:57:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     1|2025-12-07 17:57:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     2|2025-12-07 17:57:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     3|2025-12-07 17:58:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     4|2025-12-07 17:58:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     5|2025-12-07 17:58:...|            0|
|NULL|[7B 22 73 63 68 6...|sedona-debezium.p...|        1|     6|2025-12-07 17:58:...|     

In [20]:
geometry_df\
    .withColumn("geohash", f.expr("ST_GeoHash(geom, 5)"))\
    .orderBy("geohash")\
    .write\
    .mode("overwrite")\
    .format("geoparquet")\
    .save("s3a://apache-sedona-book-local/postgis-cdc-batch")
